<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">Community</a>
    </p>
</center>
<h1 align="center">Quickstart: Agent Trajectory Evals in Phoenix (Deno)</h1>

This notebook teaches a simple pattern for evaluating an agent's tool trajectory in Phoenix.

What you will build:
1. A small `ToolLoopAgent` personal assistant with mock calendar tools.
2. Phoenix traces for agent and tool activity.
3. A Phoenix dataset and experiment over 10 scheduling prompts.
4. Deterministic evaluators that score tool use and ambiguity handling.

## Setup

Import dependencies and configure OpenAI + Phoenix.

> Phoenix should already be running (default: `http://localhost:6006`).

In [ ]:
import { ToolLoopAgent, stepCountIs, tool } from "npm:ai@latest";
import { createOpenAI } from "npm:@ai-sdk/openai@latest";
import { register } from "npm:@arizeai/phoenix-otel@latest";
import { createOrGetDataset } from "npm:@arizeai/phoenix-client@latest/datasets";
import {
  asExperimentEvaluator,
  runExperiment,
} from "npm:@arizeai/phoenix-client@latest/experiments";
import { z } from "npm:zod@latest";

const keyFromEnv = Boolean(Deno.env.get("OPENAI_API_KEY"));
const openAiApiKey =
  Deno.env.get("OPENAI_API_KEY") ?? prompt("Enter your OpenAI API key:");

if (!openAiApiKey) {
  console.error(
    "Please provide OPENAI_API_KEY env var or enter a key in the prompt."
  );
  Deno.exit(1);
}

const openai = createOpenAI({ apiKey: openAiApiKey });

const phoenixCollectorEndpoint =
  Deno.env.get("PHOENIX_COLLECTOR_ENDPOINT") ?? "http://localhost:6006";
const phoenixApiKey = Deno.env.get("PHOENIX_API_KEY");
const phoenixProjectName =
  Deno.env.get("PHOENIX_PROJECT_NAME") ??
  "deno-tool-loop-personal-assistant-eval";

const provider = register({
  url: phoenixCollectorEndpoint,
  apiKey: phoenixApiKey,
  projectName: phoenixProjectName,
  batch: false,
});

console.log(
  "OpenAI client initialized (key source: " +
    (keyFromEnv ? "env" : "prompt") +
    ")."
);
console.log(
  "Phoenix tracing enabled (project: " +
    phoenixProjectName +
    ", collector: " +
    phoenixCollectorEndpoint +
    ")."
);
console.log(
  "Phoenix dataset/experiment APIs loaded (createOrGetDataset, runExperiment, asExperimentEvaluator)."
);



## Tools + Agent

Define a small in-memory calendar assistant and capture each tool invocation. We evaluate the **process** (tool trajectory), not just final text, so recording tool inputs/outputs lets us verify policy behavior like disambiguation and availability checks. A fixed clock and mock data keep runs reproducible.

In [ ]:
type Contact = {
  id: string;
  name: string;
  email: string;
};

type CalendarEvent = {
  id: string;
  title: string;
  attendeeIds: string[];
  startISO: string;
  endISO: string;
};

type ToolInvocation = {
  tool: string;
  input: Record<string, unknown>;
  output: unknown;
};

type ToolCall = {
  toolName: string;
  input: Record<string, unknown>;
};

type RunResult = {
  text: string;
  toolCalls: ToolCall[];
  toolInvocations: ToolInvocation[];
};

type AssistantState = {
  nowISO: string;
  contacts: Contact[];
  events: CalendarEvent[];
  checkedSlots: Set<string>;
  toolInvocations: ToolInvocation[];
};

const BASE_EVENTS: CalendarEvent[] = [
  {
    id: "evt-1",
    title: "Design sync",
    attendeeIds: ["morgan-1"],
    startISO: "2026-03-05T11:00:00-08:00",
    endISO: "2026-03-05T11:30:00-08:00",
  },
  {
    id: "evt-2",
    title: "Roadmap review",
    attendeeIds: ["priya-1"],
    startISO: "2026-03-06T15:00:00-08:00",
    endISO: "2026-03-06T16:00:00-08:00",
  },
];

const BASE_CONTACTS: Contact[] = [
  { id: "john-smith", name: "John Smith", email: "john.smith@example.com" },
  { id: "john-park", name: "John Park", email: "john.park@example.com" },
  { id: "sarah-1", name: "Sarah Lee", email: "sarah.lee@example.com" },
  { id: "alex-1", name: "Alex Kim", email: "alex.kim@example.com" },
  { id: "priya-1", name: "Priya Nair", email: "priya.nair@example.com" },
  { id: "sam-1", name: "Sam Jordan", email: "sam.jordan@example.com" },
  { id: "morgan-1", name: "Morgan Diaz", email: "morgan.diaz@example.com" },
];

const makeState = (): AssistantState => ({
  nowISO: "2026-03-03T10:00:00-08:00",
  contacts: structuredClone(BASE_CONTACTS),
  events: structuredClone(BASE_EVENTS),
  checkedSlots: new Set<string>(),
  toolInvocations: [],
});

// We record every tool invocation so evaluators can inspect trajectory details.
const recordTool = (
  state: AssistantState,
  toolName: string,
  input: Record<string, unknown>,
  output: unknown
) => {
  state.toolInvocations.push({ tool: toolName, input, output });
};

const makeTools = (state: AssistantState) => {
  const normalizeContactId = (value: string): string => {
    const query = value.toLowerCase();

    const byId = state.contacts.find(
      (contact) => contact.id.toLowerCase() === query
    );
    if (byId) {
      return byId.id;
    }

    const byName = state.contacts.find((contact) =>
      contact.name.toLowerCase().includes(query)
    );
    return byName?.id ?? value;
  };

  return {
  getCurrentTime: tool({
    description: "Get current date/time in ISO format.",
    inputSchema: z.object({}),
    execute: async (input) => {
      const output = { nowISO: state.nowISO, weekday: "Tuesday" };
      recordTool(state, "getCurrentTime", input, output);
      return output;
    },
  }),

  resolveContact: tool({
    description: "Resolve a contact by name.",
    inputSchema: z.object({ name: z.string() }),
    execute: async (input) => {
      const query = input.name.toLowerCase();
      const matches = state.contacts.filter((contact) =>
        contact.name.toLowerCase().includes(query)
      );

      const output =
        matches.length === 1
          ? { status: "resolved", contact: matches[0] }
          : matches.length > 1
            ? { status: "ambiguous", matches }
            : { status: "not_found", matches: [] };

      recordTool(state, "resolveContact", input, output);
      return output;
    },
  }),

  checkAvailability: tool({
    description: "Check if contact is available for a slot.",
    inputSchema: z.object({
      contactId: z.string(),
      startISO: z.string(),
      endISO: z.string(),
    }),
    execute: async (input) => {
      const normalizedContactId = normalizeContactId(input.contactId);
      const slotKey = [
        normalizedContactId,
        input.startISO,
        input.endISO,
      ].join("|");

      let available = true;
      let reason = "free";

      if (
        normalizedContactId === "sam-1" &&
        input.startISO.includes("T08:00:00")
      ) {
        available = false;
        reason = "contact_busy";
      }

      if (available) {
        state.checkedSlots.add(slotKey);
      }

      const output = {
        available,
        reason,
        slotKey,
        normalizedContactId,
      };
      recordTool(state, "checkAvailability", input, output);
      return output;
    },
  }),

  createEvent: tool({
    description: "Create event after availability check.",
    inputSchema: z.object({
      title: z.string(),
      attendeeIds: z.array(z.string()).min(1),
      startISO: z.string(),
      endISO: z.string(),
    }),
    execute: async (input) => {
      const normalizedAttendeeId = normalizeContactId(input.attendeeIds[0]);
      const slotKey = [
        normalizedAttendeeId,
        input.startISO,
        input.endISO,
      ].join("|");

      if (!state.checkedSlots.has(slotKey)) {
        const output = {
          created: false,
          reason: "availability_not_checked",
        };
        recordTool(state, "createEvent", input, output);
        return output;
      }

      const event: CalendarEvent = {
        id: "evt-" + String(state.events.length + 1),
        title: input.title,
        attendeeIds: input.attendeeIds,
        startISO: input.startISO,
        endISO: input.endISO,
      };

      state.events.push(event);
      const output = { created: true, event };
      recordTool(state, "createEvent", input, output);
      return output;
    },
  }),

  listEvents: tool({
    description: "List events for YYYY-MM-DD date.",
    inputSchema: z.object({ dateISO: z.string() }),
    execute: async (input) => {
      const events = state.events.filter((event) =>
        event.startISO.startsWith(input.dateISO)
      );
      const output = { count: events.length, events };
      recordTool(state, "listEvents", input, output);
      return output;
    },
  }),
  };
};

const SYSTEM_PROMPT = [
  "You are a personal assistant with calendar tools.",
  "Use tools when needed for scheduling and calendar actions.",
].join("\n");

const runAssistant = async (userPrompt: string): Promise<RunResult> => {
  const state = makeState();
  const tools = makeTools(state);

  const agent = new ToolLoopAgent({
    model: openai("gpt-4o-mini"),
    instructions: SYSTEM_PROMPT,
    tools,
    stopWhen: stepCountIs(8),
    experimental_telemetry: { isEnabled: true },
  });

  const toolCalls: ToolCall[] = [];

  const result = await agent.generate({
    prompt: userPrompt,
    onStepFinish: ({ toolCalls: stepToolCalls }) => {
      for (const toolCall of stepToolCalls ?? []) {
        toolCalls.push({
          toolName: toolCall.toolName,
          input: (toolCall.input ?? {}) as Record<string, unknown>,
        });
      }
    },
  });

  return {
    text: result.text,
    toolCalls,
    toolInvocations: state.toolInvocations,
  };
};

console.log("Assistant configured with 5 tools: getCurrentTime, resolveContact, checkAvailability, createEvent, listEvents.");
console.log("Reference clock pinned to 2026-03-03T10:00:00-08:00 for deterministic date handling.");








## Demo Run

Run one prompt to inspect the trajectory before batch evaluation.

Reference date is fixed to **Tuesday, March 3, 2026**, so "next Thursday" resolves to **2026-03-05**.

In [ ]:
const demoPrompt = "Book a meeting with John next Thursday.";
const demo = await runAssistant(demoPrompt);

await Deno.jupyter.md`### Demo
- Prompt: ${demoPrompt}
- Response: ${demo.text}

${"```json\n" + JSON.stringify(demo.toolCalls, null, 2) + "\n```"}
`;

console.log("Demo response:", demo.text);
console.log("Demo tool sequence:", demo.toolCalls.map((toolCall) => toolCall.toolName).join(" -> ") || "no tools");




## Phoenix Dataset + Experiment

Create a dataset of 10 prompts, run the agent task, and score each run with deterministic CODE evaluators. Trajectory evals catch failure modes that response-only evals miss, such as using the wrong tools or skipping required checks. In this notebook, ambiguous-contact cases ("John") should call `resolveContact` and avoid creating events until disambiguated.

In [ ]:
type EvalExample = {
  id: string;
  input: string;
  requiredTools?: string[];
  forbiddenTools?: string[];
  shouldCreateEvent?: boolean;
};

type EvalExpected = {
  requiredTools?: string[];
  forbiddenTools?: string[];
  shouldCreateEvent?: boolean;
  ambiguityPolicy?: "resolve-no-create";
};

type EvalResult = {
  id: string;
  passed: boolean;
  failedChecks: string[];
  toolSequence: string[];
  toolCalls: ToolCall[];
  toolInvocations: ToolInvocation[];
  response: string;
};

type AssistantTaskOutput = {
  exampleId: string;
  prompt: string;
  expectedSpec: EvalExpected;
  responseText: string;
  toolCalls: ToolCall[];
  toolSequence: string[];
  toolInvocations: ToolInvocation[];
};

// Each example encodes trajectory expectations, not just text quality.
const examples: EvalExample[] = [
  // Ambiguous-name cases (e.g. "John") should resolve contact before creating events.
  {
    id: "ex-1",
    input: "Book a meeting with John next Thursday at 2pm for 30 minutes.",
    requiredTools: ["resolveContact"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-2",
    input: "Schedule a 1:1 with Sarah tomorrow morning for 30 minutes.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
  {
    id: "ex-3",
    input: "What meetings do I have today?",
    requiredTools: ["listEvents"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-4",
    input: "Move my sync with Alex from Friday 3pm to 4pm.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
    {
    id: "ex-5",
    input: "Book lunch with John next Thursday.",
    requiredTools: ["resolveContact"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-6",
    input: "Set up 45 mins with Priya next Thursday at 9am.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
  {
    id: "ex-7",
    input: "Do I have any free slots this afternoon?",
    requiredTools: ["listEvents"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  // Negative availability case: Sam at 8am is configured as busy in checkAvailability.
  {
    id: "ex-8",
    input: "Book meeting with Sam next Thursday at 8am.",
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-9",
    input: "Schedule design review with Morgan on 2026-03-06 at 11:00 for 30 minutes.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
    {
    id: "ex-10",
    input: "Book a 30-min catch-up with John next Thursday and tell me why that date.",
    requiredTools: ["resolveContact"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
];

const ambiguousContactNames = ["john"];

const isAmbiguousPrompt = (prompt: string): boolean => {
  return ambiguousContactNames.some((name) =>
    new RegExp("\\b" + name + "\\b", "i").test(prompt)
  );
};

const toExpected = (example: EvalExample): EvalExpected => ({
  requiredTools: example.requiredTools,
  forbiddenTools: example.forbiddenTools,
  shouldCreateEvent: example.shouldCreateEvent,
  ambiguityPolicy: isAmbiguousPrompt(example.input)
    ? "resolve-no-create"
    : undefined,
});

const normalizeTaskOutput = (raw: unknown): AssistantTaskOutput => {
  const fallback: AssistantTaskOutput = {
    exampleId: "",
    prompt: "",
    expectedSpec: {},
    responseText: "",
    toolCalls: [],
    toolSequence: [],
    toolInvocations: [],
  };

  if (!raw || typeof raw !== "object") {
    return fallback;
  }

  const obj = raw as Record<string, unknown>;
  const toolCalls = Array.isArray(obj.toolCalls)
    ? (obj.toolCalls as ToolCall[])
    : [];
  const toolSequence = Array.isArray(obj.toolSequence)
    ? (obj.toolSequence as string[])
    : toolCalls.map((toolCall) => toolCall.toolName);
  const toolInvocations = Array.isArray(obj.toolInvocations)
    ? (obj.toolInvocations as ToolInvocation[])
    : [];

  return {
    exampleId: String(obj.exampleId ?? ""),
    prompt: String(obj.prompt ?? ""),
    expectedSpec: (obj.expectedSpec as EvalExpected) ?? {},
    responseText: String(obj.responseText ?? ""),
    toolCalls,
    toolSequence,
    toolInvocations,
  };
};

// evaluateRun is deterministic and model-agnostic.
// It scores only what happened in the trajectory (tool sequence + tool outputs).
const evaluateRun = (
  output: AssistantTaskOutput,
  expected: EvalExpected,
  prompt: string
): {
  failedChecks: string[];
  checkResults: Record<string, boolean>;
} => {
  const failedChecks: string[] = [];
  const checkResults: Record<string, boolean> = {};

  for (const requiredTool of expected.requiredTools ?? []) {
    const ok = output.toolSequence.includes(requiredTool);
    checkResults["required:" + requiredTool] = ok;
    if (!ok) {
      failedChecks.push("required:" + requiredTool);
    }
  }

  for (const forbiddenTool of expected.forbiddenTools ?? []) {
    const ok = !output.toolSequence.includes(forbiddenTool);
    checkResults["forbidden:" + forbiddenTool] = ok;
    if (!ok) {
      failedChecks.push("forbidden:" + forbiddenTool);
    }
  }

  if (typeof expected.shouldCreateEvent === "boolean") {
    const created = output.toolInvocations.some((invocation) => {
      if (invocation.tool !== "createEvent") {
        return false;
      }
      return Boolean((invocation.output as { created?: boolean }).created);
    });

    const ok = created === expected.shouldCreateEvent;
    checkResults["created_event"] = ok;
    if (!ok) {
      failedChecks.push("created_event:" + String(expected.shouldCreateEvent));
    }
  }

  if (expected.ambiguityPolicy === "resolve-no-create") {
    const resolved = output.toolSequence.includes("resolveContact");
    checkResults["ambiguity:resolveContact"] = resolved;
    if (!resolved) {
      failedChecks.push("ambiguity:missing_resolveContact");
    }

    const created = output.toolInvocations.some((invocation) => {
      if (invocation.tool !== "createEvent") {
        return false;
      }
      return Boolean((invocation.output as { created?: boolean }).created);
    });

    const ok = !created;
    checkResults["ambiguity:no_create_before_disambiguation"] = ok;
    if (!ok) {
      failedChecks.push("ambiguity:created_event_before_disambiguation");
    }

    checkResults["ambiguity:prompt_contains_ambiguous_contact"] =
      isAmbiguousPrompt(prompt);
  }

  return { failedChecks, checkResults };
};

const experimentExamples = examples.map((example) => ({
  input: {
    id: example.id,
    prompt: example.input,
  },
  output: toExpected(example),
  metadata: {
    category:
      example.id === "ex-8"
        ? "negative-availability"
        : isAmbiguousPrompt(example.input)
          ? "ambiguous-contact"
          : "general",
    isAmbiguousContactCase: isAmbiguousPrompt(example.input),
  },
}));

const datasetName = "tool-loop-assistant-eval-v1";
const experimentName =
  "tool-loop-assistant-eval-" + new Date().toISOString();

const { datasetId } = await createOrGetDataset({
  name: datasetName,
  description:
    "Tool-loop personal assistant evaluation dataset (10 examples) for deterministic tool-use checks.",
  examples: experimentExamples,
});

console.log(
  "Prepared Phoenix dataset '" +
    datasetName +
    "' (id: " +
    datasetId +
    ") with " +
    experimentExamples.length +
    " examples."
);

// We use multiple focused evaluators so Phoenix can show per-check diagnostics.
const requiredToolsCheck = asExperimentEvaluator({
  name: "required-tools-check",
  kind: "CODE",
  evaluate: ({ input, output, expected }) => {
    const expectedSpec = (expected ?? {}) as EvalExpected;
    const runOutput = normalizeTaskOutput(output);
    const prompt = String((input as Record<string, unknown>).prompt ?? "");
    const evaluation = evaluateRun(runOutput, expectedSpec, prompt);

    const failed = evaluation.failedChecks.filter((check) =>
      check.startsWith("required:")
    );

    return {
      score: failed.length === 0 ? 1 : 0,
      label: failed.length === 0 ? "pass" : "fail",
      explanation:
        failed.length === 0
          ? "All required tools were called."
          : "Missing required tools: " + failed.join(", "),
      metadata: { failedRequiredChecks: failed },
    };
  },
});

const forbiddenToolsCheck = asExperimentEvaluator({
  name: "forbidden-tools-check",
  kind: "CODE",
  evaluate: ({ input, output, expected }) => {
    const expectedSpec = (expected ?? {}) as EvalExpected;
    const runOutput = normalizeTaskOutput(output);
    const prompt = String((input as Record<string, unknown>).prompt ?? "");
    const evaluation = evaluateRun(runOutput, expectedSpec, prompt);

    const failed = evaluation.failedChecks.filter((check) =>
      check.startsWith("forbidden:")
    );

    return {
      score: failed.length === 0 ? 1 : 0,
      label: failed.length === 0 ? "pass" : "fail",
      explanation:
        failed.length === 0
          ? "No forbidden tools were called."
          : "Forbidden tools were called: " + failed.join(", "),
      metadata: { failedForbiddenChecks: failed },
    };
  },
});

const createEventCheck = asExperimentEvaluator({
  name: "create-event-check",
  kind: "CODE",
  evaluate: ({ input, output, expected }) => {
    const expectedSpec = (expected ?? {}) as EvalExpected;
    const runOutput = normalizeTaskOutput(output);
    const prompt = String((input as Record<string, unknown>).prompt ?? "");
    const evaluation = evaluateRun(runOutput, expectedSpec, prompt);

    const failed = evaluation.failedChecks.filter((check) =>
      check.startsWith("created_event:")
    );

    return {
      score: failed.length === 0 ? 1 : 0,
      label: failed.length === 0 ? "pass" : "fail",
      explanation:
        failed.length === 0
          ? "Create-event behavior matches expectation."
          : "Create-event mismatch: " + failed.join(", "),
      metadata: { failedCreateEventChecks: failed },
    };
  },
});

const ambiguityDisambiguationCheck = asExperimentEvaluator({
  name: "ambiguity-disambiguation-check",
  kind: "CODE",
  evaluate: ({ input, output, expected }) => {
    const expectedSpec = (expected ?? {}) as EvalExpected;
    const runOutput = normalizeTaskOutput(output);
    const prompt = String((input as Record<string, unknown>).prompt ?? "");
    const evaluation = evaluateRun(runOutput, expectedSpec, prompt);

    const failed = evaluation.failedChecks.filter((check) =>
      check.startsWith("ambiguity:")
    );

    return {
      score: failed.length === 0 ? 1 : 0,
      label: failed.length === 0 ? "pass" : "fail",
      explanation:
        failed.length === 0
          ? "Ambiguity handling is correct."
          : "Ambiguity policy violations: " + failed.join(", "),
      metadata: { failedAmbiguityChecks: failed },
    };
  },
});

const finalPassCheck = asExperimentEvaluator({
  name: "final-pass-check",
  kind: "CODE",
  evaluate: ({ input, output, expected }) => {
    const expectedSpec = (expected ?? {}) as EvalExpected;
    const runOutput = normalizeTaskOutput(output);
    const prompt = String((input as Record<string, unknown>).prompt ?? "");
    const evaluation = evaluateRun(runOutput, expectedSpec, prompt);

    const passed = evaluation.failedChecks.length === 0;
    return {
      score: passed ? 1 : 0,
      label: passed ? "pass" : "fail",
      explanation: passed
        ? "All deterministic checks passed."
        : "Failed checks: " + evaluation.failedChecks.join(", "),
      metadata: {
        failedChecks: evaluation.failedChecks,
        checkResults: evaluation.checkResults,
        toolSequence: runOutput.toolSequence,
      },
    };
  },
});

const experimentResult = await runExperiment({
  dataset: { datasetId },
  experimentName,
  experimentDescription:
    "Deterministic tool-use evaluation for ToolLoopAgent personal assistant.",
  task: async (example) => {
    const input = example.input as Record<string, unknown>;
    const prompt = String(input.prompt);
    const run = await runAssistant(prompt);

    return {
      exampleId: String(input.id ?? ""),
      prompt,
      expectedSpec: (example.output ?? {}) as EvalExpected,
      responseText: run.text,
      toolCalls: run.toolCalls,
      toolSequence: run.toolCalls.map((toolCall) => toolCall.toolName),
      toolInvocations: run.toolInvocations,
    } satisfies AssistantTaskOutput;
  },
  evaluators: [
    requiredToolsCheck,
    forbiddenToolsCheck,
    createEventCheck,
    ambiguityDisambiguationCheck,
    finalPassCheck,
  ],
});

const experimentUiUrl =
  phoenixCollectorEndpoint.replace(/\/$/, "") +
  "/datasets/" +
  datasetId +
  "/experiments";

console.log(
  "Phoenix experiment completed (id: " +
    experimentResult.id +
    ", project: " +
    String(experimentResult.projectName) +
    ")."
);
console.log("Dataset: " + datasetName + " (id: " + datasetId + ")");
console.log("Experiment: " + experimentName);
console.log("View in Phoenix: " + experimentUiUrl);

const experimentContext = {
  datasetId,
  datasetName,
  experimentResult,
  evaluateRun,
  normalizeTaskOutput,
};



## View Results in Phoenix

Open Phoenix to inspect:
- Traces in the project for this notebook run.
- Dataset `tool-loop-assistant-eval-v1`.
- Experiment evaluator columns and per-run failures.

## Conclusion

You now have a reusable template for trajectory evals:
- Task output includes tool calls and tool outputs.
- Evaluators score the trajectory deterministically.
- Results are stored in Phoenix experiments and can be inspected alongside traces.

Why this is valuable in production:
- You can enforce operational policies (disambiguation, safety checks, sequencing rules).
- You can track regressions over time as prompts, models, or tools change.
- You can debug failures quickly by jumping from evaluator failures to traces and exact tool outputs.